<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/python-primer-notebooks/blob/main/17-categories.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/python-primer/) collection._

# Chapter 17 — Categories and Encoding

Companion to [the chapter](https://www.ai.biz/books/python-primer/categories-and-encoding/).


In [ ]:
import pandas as pd, numpy as np
rng = np.random.default_rng(3)


## 1. The categorical dtype


In [ ]:
s = pd.Series(['gold','silver','bronze'] * 100_000)
as_obj = s.memory_usage(deep=True)
as_cat = s.astype('category').memory_usage(deep=True)
print(f'object   : {as_obj:>12,} bytes')
print(f'category : {as_cat:>12,} bytes')
print(f'reduction: {as_obj/as_cat:.0f}x')


## 2. Ordered categories make sorting and comparison work


In [ ]:
from pandas.api.types import CategoricalDtype
sizes = CategoricalDtype(['small','medium','large'], ordered=True)
d = pd.DataFrame({'size': pd.Series(['large','small','medium']).astype(sizes)})
print('sorted correctly:'); print(d.sort_values('size'))
print()
print('comparison works:', (d['size'] > 'small').tolist())


## 3. The trap: filtering keeps unused categories


In [ ]:
df = pd.DataFrame({'tier': pd.Series(rng.choice(['gold','silver','bronze'], 300)).astype('category'),
                   'revenue': rng.random(300)*100})
sub = df[df.tier == 'gold']
print('categories still listed:', list(sub.tier.cat.categories))
print()
print('groupby produces empty rows:'); print(sub.groupby('tier', observed=False).size())
print()
print('with observed=True:'); print(sub.groupby('tier', observed=True).size())


## 4. One-hot encoding


In [ ]:
from sklearn.preprocessing import OneHotEncoder
enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X = pd.DataFrame({'city': ['delhi','tokyo','sydney','delhi']})
enc.fit(X)
print('categories learned:', enc.categories_[0])
print()
unseen = pd.DataFrame({'city': ['lagos']})
print('unseen city encodes to all zeros, no exception:')
print(enc.transform(unseen))


## 5. Frequency encoding — one column, any cardinality


In [ ]:
cities = pd.Series(rng.choice(['delhi','tokyo','sydney','rare'], 1000, p=[.5,.3,.19,.01]))
freq = cities.value_counts(normalize=True)
print(freq.round(3))
print()
print('encoded:', cities.map(freq).head(5).round(3).tolist())


## 6. Target encoding leaks — demonstrated


In [ ]:
n = 600
d = pd.DataFrame({'city': rng.choice([f'c{i}' for i in range(200)], n),   # high cardinality
                  'y': rng.integers(0, 2, n)})                            # PURE NOISE

# WRONG: each row's encoding uses its own target
means = d.groupby('city').y.mean()
d['leaky'] = d.city.map(means)

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

auc = cross_val_score(LogisticRegression(), d[['leaky']], d.y,
                      cv=5, scoring='roc_auc').mean()
print(f'AUC using naive target encoding on PURE NOISE: {auc:.3f}')
print('There is no signal in this data. The feature contains the answer.')


## 7. Out-of-fold encoding fixes it


In [ ]:
from sklearn.model_selection import KFold

def target_encode_oof(df, col, target, n_splits=5, smoothing=10):
    gmean = df[target].mean()
    enc = pd.Series(np.nan, index=df.index)
    for tr, va in KFold(n_splits, shuffle=True, random_state=0).split(df):
        stats = df.iloc[tr].groupby(col)[target].agg(['mean','count'])
        sm = (stats['mean']*stats['count'] + gmean*smoothing) / (stats['count'] + smoothing)
        enc.iloc[va] = df.iloc[va][col].map(sm).values
    return enc.fillna(gmean)

d['honest'] = target_encode_oof(d, 'city', 'y')
auc2 = cross_val_score(LogisticRegression(), d[['honest']], d.y,
                       cv=5, scoring='roc_auc').mean()
print(f'AUC with out-of-fold encoding: {auc2:.3f}  <- correctly near 0.5')


Near 0.5 is the truth: there was never any signal. The first number was the feature reading the answer.


## 8. Group the rare tail before reaching for anything clever


In [ ]:
top = cities.value_counts().nlargest(3).index
grouped = cities.where(cities.isin(top), 'other')
print(f'{cities.nunique()} categories -> {grouped.nunique()}')
print(grouped.value_counts())


## Try it yourself

1. Raise `smoothing` to 100 and watch the encoded values collapse toward the global mean.
2. Reduce the city count to 5 and see how much less the naive version leaks.
3. Encode an ordered size column both ordinally and one-hot, and compare with a linear model.
